# B3 exploration · Who owns the next step, the history and the action?

Start with the [B3 snippets](../snippets/b3/README.md). Explore prescribed
routing, an agent's actual tool choices, execution limits, specialist ownership,
conversation stores, application task data and a genuinely paused SDK tool call.

Default Run All executes deterministic routing/storage experiments and labels
inference SKIPPED. Enable `RUN_LIVE` for real model calls. Optional delegation
has its own switch. All operational facts and schedules are synthetic. The
only action appends to an **in-memory demonstration outbox**; it sends nothing.

This notebook supplies complete experiments. The
[B3 TODO lab](../labs/container_help_desk/B3.md) is the separate application assignment.
Use a fresh kernel and Run All before comparing changed experiments.

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

TRACK_B = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'snippets/b1').is_dir()), None)
if TRACK_B is None:
    raise RuntimeError('Open this notebook from track-b or track-b/notebooks.')
for env_file in [TRACK_B / 'notebooks/.env', TRACK_B / 'snippets/.env',
                 TRACK_B / '.env', TRACK_B.parent / '.env']:
    load_dotenv(env_file)

RUN_LIVE = False  # Change to True, then restart and Run All for actual model experiments.
MODEL = 'gpt-4.1-mini'
if RUN_LIVE and not os.environ.get('OPEN_AI_KEY'):
    raise ValueError('Configure OPEN_AI_KEY before enabling live calls.')
print('Live inference enabled:', RUN_LIVE)


from agents import Agent, Runner, function_tool, set_default_openai_key, set_tracing_disabled

set_tracing_disabled(True)
if RUN_LIVE:
    set_default_openai_key(os.environ['OPEN_AI_KEY'])

from agents import MaxTurnsExceeded, SQLiteSession, RunState

## 1. Prescribed routing has observable consequences

**Predict:** which IDs cause an events read? These are ordinary Python
conditions. A model is not choosing the route.

In [ ]:
statuses = {'C1': 'delayed', 'C2': 'on_time'}
events = {'C1': ['Customs hold', 'Planned departure missed'], 'C2': ['Departed as planned']}
scheduled_departures = {'C1': '2026-11-05T18:00:00Z'}
routing_reads = []
for container_id in ['C1', 'C2', 'C9']:
    status = statuses.get(container_id, 'unknown')
    if status == 'delayed':
        route = 'investigate'
        routing_reads.append(('events', container_id))
    elif status == 'on_time':
        route = 'finish'
    else:
        route = 'clarify'
    print(container_id, status, '->', route)
print('Events reads selected by code:', routing_reads)

**Inspect:** C2 and C9 need different outcomes, although neither needs the
same events lookup as C1. A missing record is not a recorded delay.

**Change:** make C2 delayed and rerun this cell. The route changes because
you changed a fixed condition's input, not because a model learned a new policy.
Restore the initial fixtures before comparing the later agent experiment.

## 2. Give one agent capabilities with distinct observations

These tools own their read behavior and retain their actual observations in
an application list. The agent's instruction asks it to read status first;
the later trace will show whether its choices fit the task. The schedule is
not evidence of a completed departure.

In [ ]:
observations = []

@function_tool
def read_status(container_id: str) -> dict:
    """Read a synthetic recorded status; unknown IDs have no known status."""
    row = {'container_id': container_id, 'status': statuses.get(container_id, 'unknown')}
    observations.append(('read_status', dict(row)))
    return row

@function_tool
def read_events(container_id: str) -> dict:
    """Read recorded tracking events; preserve missing evidence."""
    row = {'events': events.get(container_id), 'source': 'synthetic tracking'}
    observations.append(('read_events', dict(row)))
    return row

In [ ]:
@function_tool
def read_departure(container_id: str) -> dict:
    """Read a scheduled departure; this does not confirm actual movement."""
    row = {'scheduled_departure': scheduled_departures.get(container_id)}
    observations.append(('read_departure', dict(row)))
    return row

investigator = Agent(
    name='Investigator', model=MODEL,
    instructions=('Read recorded status first. Investigate only an observed delay, using '
                  'events and scheduled departure before drafting. For unknown IDs ask for '
                  'a corrected ID. Use only observations, preserve uncertainty and never '
                  'claim that a scheduled movement happened. Return a short draft for review.'),
    tools=[read_status, read_events, read_departure],
)

## 3. Compare the intended path with an actual investigation

**Predict:** for C1, which facts must a supported draft contain? The SDK can
finish without meeting your intended evidence requirements; inspect both
observations and final text. Change `container_id` to C2 or C9 for a controlled
variation. The model may choose a different path from the prescribed loop.

In [ ]:
container_id = 'C1'
session = SQLiteSession('investigation-' + container_id)  # This object owns an in-memory store.
investigation_result = None
observations.clear()
if RUN_LIVE:
    investigation_result = await Runner.run(
        investigator, f'Check {container_id} and prepare a concise update for review.',
        session=session, max_turns=6,
    )
    print('ACTUAL OBSERVATIONS:', observations)
    print('DRAFT:', investigation_result.final_output)
    print('Model responses:', len(investigation_result.raw_responses))
else:
    print('SKIPPED: actual agent investigation; no prepared draft is substituted.')

## 4. Conversation history and task state answer different questions

**Predict:** can generated prose saying “approved” mutate an ordinary Python
review field? The SDK session supplies conversation context. The application
owns the task dictionary and its review status.

In [ ]:
task = {'container_id': container_id, 'findings': list(observations),
        'draft': investigation_result.final_output if investigation_result else None,
        'review': 'pending'}
print('Application-owned task:', json.dumps(task, indent=2))
print('Session currently stores', len(await session.get_items()), 'conversation items')
print('Independent application review field:', task['review'])

In [ ]:
if RUN_LIVE:
    follow_up = 'Which container were we discussing? Reply with the identifier only; do not look it up.'
    for label, conversation in [('same store', session), ('fresh store, same ID', SQLiteSession(session.session_id))]:
        answer = await Runner.run(investigator, follow_up, session=conversation, max_turns=3)
        print(label, '->', answer.final_output)
else:
    print('SKIPPED: conversational-reference comparison needs a live investigation.')

**Inspect:** the two session objects above have the same identifier but their
default stores are different in-memory databases. The second object does not
magically discover the first object's history. A model might guess an ID;
that would not prove that empty storage contained it.

The task's `findings` list copied the observation list before later experiments
clear it. Rerunning the investigation does not automatically update an old
`task` variable; rerun the task-creation cell when you want a new snapshot.

## 5. Inspect storage independently of model behavior

This uses only prepared **user** text to inspect the session store. No
assistant response is fabricated. Predict whether the same ID in separate
in-memory stores contains the same items.

In [ ]:
left = SQLiteSession('same-id')
right = SQLiteSession('same-id')
await left.add_items([{'role': 'user', 'content': 'Prepared storage example, not a model reply.'}])
print('First store:', await left.get_items())
print('Different in-memory store:', await right.get_items())
left.close()
right.close()

**Next prediction:** if both objects point at one file and use the same ID,
what changes? This follow-on temporarily shares a backing store, then removes
it. It demonstrates conversation persistence, not a paused agent runtime.

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    database = Path(directory) / 'conversation.db'
    saved_session = SQLiteSession('shared-id', db_path=database)
    await saved_session.add_items([{'role': 'user', 'content': 'Prepared persisted user message.'}])
    reopened_session = SQLiteSession('shared-id', db_path=database)
    print('Reopened shared store:', await reopened_session.get_items())
    saved_session.close()
    reopened_session.close()

## 6. Serializing task data does not create runtime state

**Predict:** what does the JSON below retain, and what is missing if you need
to resume an interrupted tool call? A saved task can retain evidence and the
actual draft. It has no pending SDK call, approvals or execution cursor.

In [ ]:
task_json = json.dumps(task)
restored_task = json.loads(task_json)
print('Restored task keys:', list(restored_task))
print('Draft exists:', restored_task['draft'] is not None)
print('Review field:', restored_task['review'])
print('Original/restored are separate dictionaries:', restored_task is not task)

## 7. A turn limit is not a total action budget

**Predict:** can a tool execute before a run reports `MaxTurnsExceeded`?
A model turn can request multiple tools. An incomplete run does not roll back
reads or side effects that already happened. This experiment uses read tools only.

In [ ]:
observations.clear()
if RUN_LIVE:
    try:
        limited = await Runner.run(investigator, 'Read C1 status and investigate its delay.', max_turns=1)
        print('Runtime returned:', limited.final_output)
    except MaxTurnsExceeded:
        print('INCOMPLETE: the model-turn budget was exhausted.')
    print('Tool work that already occurred:', observations)
else:
    print('SKIPPED: budget experiment needs actual model decisions.')

**Inspect:** count the actual work rather than assuming one turn means one
tool call. A finite turn budget is not a global token, cost, wall-clock or
nested-specialist budget. The earlier saved `task` stays an earlier snapshot;
this incomplete experiment has not silently replaced its draft.

## 8. Optional · Specialist as a tool versus conversation handoff

**Predict:** who will own the final user-facing answer? Both arrangements
below use the same model and a narrowly scoped charges specialist. Its fee is
a supplied synthetic amount, not a PortOps invoice.

In [ ]:
@function_tool
def read_charge(container_id: str) -> dict:
    """Read a synthetic fee, not a real tariff or invoice."""
    return {'usd': 100, 'source': 'illustrative fixture'} if container_id == 'C1' else {'error': 'unknown ID'}

charges = Agent(name='Charges specialist', model=MODEL,
                instructions='Use read_charge; preserve USD and label the fee illustrative.',
                handoff_description='Answers illustrative charge questions.', tools=[read_charge])
coordinator = Agent(name='Coordinator', model=MODEL,
                    instructions='Use the charges specialist, then own the final answer.',
                    tools=[charges.as_tool('assess_charge', 'Obtain an illustrative charge.', max_turns=4)])
triage = Agent(name='Triage', model=MODEL,
               instructions='Hand charge questions to the charges specialist to answer.',
               handoffs=[charges])

In [ ]:
RUN_DELEGATION = False
if RUN_LIVE and RUN_DELEGATION:
    for agent in [coordinator, triage]:
        delegated = await Runner.run(agent, "What is C1's illustrative charge in USD?", max_turns=5)
        print('Starting agent:', agent.name, 'Final owner:', delegated.last_agent.name)
        print('Answer:', delegated.final_output)
else:
    print('SKIPPED: optional comparison needs RUN_LIVE and RUN_DELEGATION.')

**Inspect:** the coordinator's nested specialist returns a result to its
caller. A handoff changes the active agent that answers. A new name alone
does not create useful specialization. Each nested run has its own bounds;
printing an outer turn count does not measure all nested work.

**Change:** add an operations specialist using the earlier status tool. Which
arrangement should combine operations and charges into one update? Explain
why this tiny task may be simpler with one investigator.

## 9. Pause before an actual action

This section uses an explicitly **supplied synthetic draft**, isolating review
from earlier inference. It does not claim the draft was generated. Later,
replace it with your inspected `task['draft']` to connect the two mechanisms.

**Predict:** after the model requests this tool, does its body run immediately?
The decorator's `needs_approval=True` controls execution. A prompt saying
“please ask first” is not the same runtime mechanism.

In [ ]:
outbox = []

@function_tool(needs_approval=True)
def queue_customer_update(container_id: str, text: str) -> str:
    """Append a reviewed message to the in-memory demo outbox; send nothing."""
    outbox.append({'container_id': container_id, 'text': text})
    return 'Recorded in the local demonstration outbox. No message was sent.'

review_agent = Agent(
    name='Review coordinator', model=MODEL,
    instructions='Queue the supplied exact payload unchanged. Never retry a rejected action. Report the actual result.',
    tools=[queue_customer_update],
)
payload = {'container_id': 'C1', 'text': 'C1 is delayed. Its next departure is scheduled for 2026-11-05 at 18:00 UTC.'}
print('Supplied draft payload:', payload)

In [ ]:
approval_result = None
restored_run = None  # A failed fresh proposal must not expose an older pending run.
outbox.clear()  # Reset only this isolated in-memory demonstration.
if RUN_LIVE:
    candidate = await Runner.run(review_agent, 'Queue this exact payload: ' + json.dumps(payload), max_turns=4)
    if not candidate.interruptions:
        raise RuntimeError('No pending action was produced. Inspect the run before calling this a successful gate.')
    print('OUTBOX BEFORE REVIEW:', outbox)
    for item in candidate.interruptions:
        print('PENDING:', item.name, item.arguments)
        if item.name != 'queue_customer_update' or json.loads(item.arguments) != payload:
            raise ValueError('The proposed operation/payload changed. Stop and inspect it before review.')
    approval_result = candidate  # Publish only after every pending payload passed inspection.
else:
    print('SKIPPED: no real approval interruption has been produced.')

## 10. Save the right kind of state

**Predict:** can the earlier `restored_task` dictionary resume this pending
call? The SDK needs its own runtime snapshot. This cell serializes that actual
pending snapshot and restores it using the same agent/tool definitions.
It makes no additional inference request.

The JSON is kept in memory here. Writing it to a file is a storage extension;
recovering arbitrary external service state is a separate problem.

In [ ]:
restored_run = None
if approval_result is not None:
    checkpoint = approval_result.to_state().to_json()
    checkpoint_copy = json.loads(json.dumps(checkpoint))
    restored_run = await RunState.from_json(review_agent, checkpoint_copy)
    print('Restored SDK pending calls:', len(restored_run.get_interruptions()))
    approval_result = None  # Consume this candidate; do not restore the same proposal twice.
    print('Task data type:', type(restored_task).__name__)
    print('Runtime state type:', type(restored_run).__name__)
else:
    print('SKIPPED: no actual pending SDK state exists to serialize.')

## 11. Apply a decision to the exact pending calls

Read the PENDING payload above before changing `DECISION`. The default is
rejection, so Run All does not approve an action. To compare approval and
rejection, rerun the proposal and snapshot cells to create a **fresh** pending
run and empty local outbox; then choose the other decision here.

Invalid input must stop before applying approval. A changed proposal after
resumption needs a new review, not an automatic second approval.

In [ ]:
DECISION = 'reject'  # Inspect PENDING before changing this to 'approve'.
if restored_run is not None:
    if DECISION not in {'approve', 'reject'}:
        raise ValueError('Expected approve or reject; no action was authorized.')
    pending_calls = list(restored_run.get_interruptions())
    for item in pending_calls:
        if item.name != 'queue_customer_update' or json.loads(item.arguments) != payload:
            raise ValueError('Pending payload no longer matches; create and inspect a fresh proposal.')
    if not pending_calls:
        raise ValueError('No pending call remains; create a fresh proposal.')
    for item in pending_calls:
        if DECISION == 'approve':
            restored_run.approve(item)
        else:
            restored_run.reject(item)
    state_to_resume, restored_run = restored_run, None  # Prevent accidental reuse of this cell.
    resumed = await Runner.run(review_agent, state_to_resume, max_turns=4)
    print('OUTBOX AFTER REVIEW:', outbox)
    print('AGENT REPLY:', resumed.final_output)
    if resumed.interruptions:
        print('STOP: another proposal needs its own review; it was not approved here.')
else:
    print('SKIPPED: no pending runtime state is available for review.')

**Inspect:** approval permits the tool body to execute; rejection leaves its
proposed effect unexecuted. Check the actual outbox instead of treating the
agent's final wording as proof. Merely serializing state does not authorize
anything. Reusing or retrying actions in a production system needs additional
identity, transaction and duplicate-action controls.

## Explain what belongs where

| Question | Evidence to inspect |
| --- | --- |
| Which tools executed? | Actual observations / runtime items |
| Which container did the chat discuss? | History in the correct session store |
| What draft awaits review? | Application task data |
| Which operation is paused? | SDK interruption and runtime state |
| What actually happened after the decision? | Outbox side effect and resumed outcome |

Transfer the mechanisms into the [B3 TODO lab](../labs/container_help_desk/B3.md)
or the [PortOps lab](../../project-supplychain-sandbox-PortOps/labs/ai_container_help_desk/README.md).
PortOps supplies different capabilities and status labels; do not copy C1's
tracking/schedule fixtures into it as if its API had returned them.

Implementation references: [running agents](https://developers.openai.com/api/docs/guides/agents/running-agents)
and [guardrails and human review](https://developers.openai.com/api/docs/guides/agents/guardrails-approvals).